In [1]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.1 MB/s eta 0:00:00


In [2]:
import torch
import torch_geometric
from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
import numpy as np
import pywt
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pandas as pd


# # --- Data Loading and Graph Construction ---
# df = pd.read_csv("/kaggle/working/brainwave_preporcessed.csv")

# # Select channel columns that are named with digits only (e.g., "0", "1", ..., "1515")
# channel_cols = [col for col in df.columns if col.isdigit()]
# if not channel_cols:
#     raise ValueError("No channel columns found in the CSV. Please verify your column names.")

# num_channels = len(channel_cols)

import torch
import pandas as pd
from torch_geometric.data import Data

# Load the dataset
df = pd.read_csv('/kaggle/input/case-dataset/case.csv')

# Drop the last 4 columns and 'valence', 'arousal' columns
df = df.iloc[:, :-4].drop(columns=['valence', 'arousal'])

# Separate features and target
concatenated_df = df.iloc[:, 1:]  # Exclude the time column  # Adjust the path accordingly

class EEGGraphDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.num_channels = 8
        self.features_per_channel = 57
        self.labels = torch.tensor(dataframe['class_label'].values, dtype=torch.long)

    def haar_wavelet_transform(self, x):
        coeffs = pywt.wavedec(x, 'haar', level=1)
        return torch.tensor(np.concatenate(coeffs), dtype=torch.float)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        features = []
        
        for i in range(self.num_channels):
            start_idx = i * self.features_per_channel
            end_idx = start_idx + self.features_per_channel
            channel_features = row[start_idx:end_idx].values.astype(np.float32)
            #transformed_features = self.haar_wavelet_transform(channel_features)
            features.append(channel_features)
        
        x = torch.tensor(features)  # Shape: (62, 5)
        edge_index = torch_geometric.utils.dense_to_sparse(torch.ones(self.num_channels, self.num_channels))[0]
        y = self.labels[index]
        
        return Data(x=x, edge_index=edge_index, y=y)
    
    def __len__(self):
        return len(self.dataframe)

# Load dataset
dataset = EEGGraphDataset(concatenated_df)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

# Define GNN Model
import torch
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing, GCNConv, global_mean_pool
from torch_geometric.utils import add_self_loops


class HaarWaveletConv(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super(HaarWaveletConv, self).__init__(aggr='mean')
        self.lin = torch.nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index):
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        x = self.lin(x)
        return self.propagate(edge_index, x=x)

    def message(self, x_j, x_i):
        return (x_j - x_i)/(2**0.5)

    def update(self, aggr_out):
        return aggr_out


class CustomGCNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5, sim_threshold=0.7):
        super(CustomGCNN, self).__init__()

        self.dropout = dropout
        self.sim_threshold = sim_threshold

        # Haar wavelet branch
        self.haar_conv1 = HaarWaveletConv(in_channels, hidden_channels)
        self.haar_conv2 = HaarWaveletConv(hidden_channels, hidden_channels)
        self.haar_conv3 = HaarWaveletConv(hidden_channels, hidden_channels)

        # Spatial GCN branch
        self.spatial_conv1 = GCNConv(in_channels, hidden_channels)
        self.spatial_conv2 = GCNConv(hidden_channels, hidden_channels)
        self.spatial_conv3 = GCNConv(hidden_channels, hidden_channels)

        # Fully connected classification layer
        self.fc = torch.nn.Linear(hidden_channels, out_channels)

    def nash_bargaining_fusion(self, x_haar, x_spatial):
        # Check similarity for win-win agreement
        sim = F.cosine_similarity(x_haar, x_spatial, dim=1)

        # Win-win mask: where similarity is high enough
        winwin_mask = sim > self.sim_threshold

        # Nash fusion (geometric mean weighted by log utility)
        fused = torch.sqrt(torch.clamp(x_haar * x_spatial, min=1e-6))

        # If not win-win, fallback to average
        fallback = 0.5 * (x_haar + x_spatial)

        return torch.where(winwin_mask.unsqueeze(1), fused, fallback)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Haar wavelet branch
        x_haar = self.haar_conv1(x, edge_index)
        x_haar = F.relu(x_haar)
        x_haar = self.haar_conv2(x_haar, edge_index)
        x_haar = F.relu(x_haar)
        #x_haar = F.dropout(x_haar, p=self.dropout, training=self.training)

        x_haar = self.haar_conv3(x_haar, edge_index)
        x_haar = F.relu(x_haar)
        #x_haar = F.dropout(x_haar, p=self.dropout, training=self.training)

        # Spatial GCN branch
        x_spatial = self.spatial_conv1(x, edge_index)
        x_spatial = F.relu(x_spatial)
        #x_spatial = F.dropout(x_spatial, p=self.dropout, training=self.training)

        x_spatial = self.spatial_conv2(x_spatial, edge_index)
        x_spatial = F.relu(x_spatial)
        x_spatial = self.spatial_conv3(x_spatial, edge_index)
        x_spatial = F.relu(x_spatial)
        #x_spatial = F.dropout(x_spatial, p=self.dropout, training=self.training)

        # Concatenate features
        x_combined = self.nash_bargaining_fusion(x_haar, x_spatial)

        # Global pooling
        x_pooled = global_mean_pool(x_combined, batch)

        # Final classification
        x_pooled = F.dropout(x_pooled, p=self.dropout, training=self.training)
        out = self.fc(x_pooled)

        return out

    

  

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomGCNN(in_channels=57, hidden_channels=32, out_channels=len(concatenated_df['class_label'].unique())).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
criterion = torch.nn.CrossEntropyLoss()
best_val_loss = float("inf")

def train_model(train_loader, val_loader, model, optimizer, criterion, epochs=10):
    global best_val_loss
    model.train()
    for epoch in range(epochs):
        total_train_loss, total_val_loss = 0, 0
        all_train_preds, all_train_labels = [], []
        
        # Training Loop
        for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch)
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()
            all_train_preds.extend(out.argmax(dim=1).cpu().numpy())
            all_train_labels.extend(batch.y.cpu().numpy())
        
        train_acc = accuracy_score(all_train_labels, all_train_preds)
        
        # Validation Loop
        model.eval()
        all_val_preds, all_val_labels = [], []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Val Epoch {epoch+1}"):
                batch = batch.to(device)
                out = model(batch)
                loss = criterion(out, batch.y)
                total_val_loss += loss.item()
                all_val_preds.extend(out.argmax(dim=1).cpu().numpy())
                all_val_labels.extend(batch.y.cpu().numpy())
        
        val_acc = accuracy_score(all_val_labels, all_val_preds)
        avg_train_loss = total_train_loss / len(train_loader)
        avg_val_loss = total_val_loss / len(val_loader)
        scheduler.step(avg_val_loss)
        
        print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.4f}, Train Acc = {train_acc:.4f}, Val Loss = {avg_val_loss:.4f}, Val Acc = {val_acc:.4f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), "best_model.pth")
            print("Saved new best model!")

def test_model(test_loader, model):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Testing"):
            batch = batch.to(device)
            out = model(batch)
            preds = out.argmax(dim=1).cpu().numpy()
            labels = batch.y.cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels)
    
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    
    print(f"Test Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")

# Train and Test the model
train_model(train_loader, test_loader, model, optimizer, criterion, epochs=10)
test_model(test_loader, model)

Training Epoch 1:   0%|          | 0/1191 [00:00<?, ?it/s]<ipython-input-2-64e36fb82625>:60: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  x = torch.tensor(features)  # Shape: (62, 5)
Val Epoch 1: 100%|██████████| 298/298 [02:10<00:00,  2.29it/s]


Epoch 1: Train Loss = 1.2885, Train Acc = 0.3864, Val Loss = 1.2027, Val Acc = 0.4575
Saved new best model!


Val Epoch 2: 100%|██████████| 298/298 [02:10<00:00,  2.28it/s]


Epoch 2: Train Loss = 1.0653, Train Acc = 0.5372, Val Loss = 0.9278, Val Acc = 0.6203
Saved new best model!


Val Epoch 3: 100%|██████████| 298/298 [02:08<00:00,  2.31it/s]


...

Epoch 9: Train Loss = 0.6813, Train Acc = 0.7351, Val Loss = 0.6657, Val Acc = 0.7440
Saved new best model!


Val Epoch 10: 100%|██████████| 298/298 [02:11<00:00,  2.27it/s]


Epoch 10: Train Loss = 0.6714, Train Acc = 0.7405, Val Loss = 0.7159, Val Acc = 0.7230


Testing: 100%|██████████| 298/298 [02:09<00:00,  2.29it/s]


Test Accuracy: 0.7230
F1 Score: 0.7232
Precision: 0.7435
Recall: 0.7230
